# Data Preparation and Preprocessing

## Notebook Objective

This notebook focuses on preparing the NASA SMAP/MSL anomaly detection dataset for modeling.

In the previous notebook, I completed data understanding and exploratory data analysis. I inspected the dataset structure, metadata, train/test `.npy` files, anomaly intervals, feature dimensions, and representative channels.

The goal of this notebook is to convert the raw dataset into clean, model-ready inputs.

## Main Tasks

This notebook will cover:

1. Load the metadata and raw `.npy` files.
2. Parse anomaly intervals and anomaly classes.
3. Create binary anomaly labels for each test time step.
4. Handle duplicated metadata entries.
5. Validate train/test/label alignment.
6. Check for missing values, infinite values, and constant features.
7. Scale train and test data correctly.
8. Prepare cleaned arrays for baseline modeling.

## Important Preprocessing Rule

To avoid data leakage, preprocessing transformations such as scaling should be fitted only on the training data and then applied to both training and test data.

The test data should not influence the fitted preprocessing parameters.

## Expected Output

By the end of this notebook, each selected channel should have:

- `train_scaled`: preprocessed training data
- `test_scaled`: preprocessed test data
- `test_labels`: binary anomaly labels
  - `0` = normal
  - `1` = anomaly

These outputs will be used later in the baseline modeling notebook.

## Step 1: Imports and Paths

In this step, I import the required libraries and define the paths to the raw dataset folders.

The raw dataset contains:

- `train/`: training telemetry arrays
- `test/`: test telemetry arrays
- `labeled_anomalies.xlsx`: metadata containing anomaly intervals for the test files

In [1]:
import os
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler


In [3]:
DATA_DIR = "../data/raw"

train_dir = os.path.join(DATA_DIR, "train")
test_dir = os.path.join(DATA_DIR, "test")
labels_path = os.path.join(DATA_DIR, "labeled_anomalies.csv")

print("Train directory exists:", os.path.exists(train_dir))
print("Test directory exists:", os.path.exists(test_dir))
print("Labels file exists:", os.path.exists(labels_path))

Train directory exists: True
Test directory exists: True
Labels file exists: True


## Step 2: Load and Parse Metadata

The metadata file contains anomaly intervals and anomaly types for each telemetry channel.

The `anomaly_sequences` column contains anomaly intervals that refer to time steps in the corresponding test file.

The `class` column contains the anomaly type for each interval, such as `point` or `contextual`.

Both columns need to be parsed into Python lists so they can be used programmatically.

In [5]:
labels_df = pd.read_csv(labels_path)

print("Metadata shape:", labels_df.shape)
labels_df.head()

Metadata shape: (82, 5)


,chan_id,spacecraft,anomaly_sequences,class,num_values
0,P-1,SMAP,"[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]",8505
1,S-1,SMAP,"[[5300, 5747]]",[point],7331
2,E-1,SMAP,"[[5000, 5030], [5610, 6086]]","[contextual, contextual]",8516
3,E-2,SMAP,"[[5598, 6995]]",[point],8532
4,E-3,SMAP,"[[5094, 8306]]",[point],8307


In [6]:
def parse_anomaly_sequences(value):
    if isinstance(value, str):
        return ast.literal_eval(value)
    return value


def parse_class_labels(value):
    if isinstance(value, list):
        return value
    
    if pd.isna(value):
        return []
    
    if isinstance(value, str):
        value = value.strip()
        value = value.replace("[", "").replace("]", "")
        labels = [item.strip() for item in value.split(",")]
        labels = [label for label in labels if label]
        return labels
    
    return value

In [7]:
labels_df["anomaly_sequences_parsed"] = labels_df["anomaly_sequences"].apply(parse_anomaly_sequences)
labels_df["class_parsed"] = labels_df["class"].apply(parse_class_labels)

labels_df["num_anomaly_intervals"] = labels_df["anomaly_sequences_parsed"].apply(len)
labels_df["num_class_labels"] = labels_df["class_parsed"].apply(len)

labels_df[
    [
        "chan_id",
        "spacecraft",
        "anomaly_sequences",
        "anomaly_sequences_parsed",
        "class",
        "class_parsed",
        "num_values",
        "num_anomaly_intervals",
        "num_class_labels"
    ]
].head()

,chan_id,spacecraft,anomaly_sequences,anomaly_sequences_parsed,class,class_parsed,num_values,num_anomaly_intervals,num_class_labels
0,P-1,SMAP,"[[2149, 2349], [4536, 4844], [3539, 3779]]","[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]","[contextual, contextual, contextual]",8505,3,3
1,S-1,SMAP,"[[5300, 5747]]","[[5300, 5747]]",[point],[point],7331,1,1
2,E-1,SMAP,"[[5000, 5030], [5610, 6086]]","[[5000, 5030], [5610, 6086]]","[contextual, contextual]","[contextual, contextual]",8516,2,2
3,E-2,SMAP,"[[5598, 6995]]","[[5598, 6995]]",[point],[point],8532,1,1
4,E-3,SMAP,"[[5094, 8306]]","[[5094, 8306]]",[point],[point],8307,1,1


## Step 3: Create Clean Metadata for Preparation

From the data understanding notebook, channel `P-2` was identified as having duplicate metadata rows with different anomaly intervals. Since both rows refer to the same raw test file but provide different label boundaries, this creates ambiguity when generating binary labels.

For the first preprocessing version, `P-2` will be excluded temporarily to keep the label generation process unambiguous.

The duplicated channel can be revisited later by either merging the intervals, selecting one annotation, or excluding it permanently based on further guidance.

In [8]:
duplicated_channel_ids = labels_df[
    labels_df["chan_id"].duplicated(keep=False)
]["chan_id"].unique().tolist()

labels_clean_df = labels_df[
    ~labels_df["chan_id"].isin(duplicated_channel_ids)
].copy()

print("Duplicated channel IDs excluded:", duplicated_channel_ids)
print("Original metadata rows:", len(labels_df))
print("Clean metadata rows:", len(labels_clean_df))
print("Unique channels in clean metadata:", labels_clean_df["chan_id"].nunique())

Duplicated channel IDs excluded: ['P-2']
Original metadata rows: 82
Clean metadata rows: 80
Unique channels in clean metadata: 80


## Step 4: Helper Functions

In this step, I create reusable functions for loading channel data, retrieving channel metadata, creating binary test labels, and checking alignment.

These functions will allow the same preparation logic to be applied to any selected channel.

In [10]:
def get_channel_metadata(labels_df, channel_id):
    channel_rows = labels_df[labels_df["chan_id"] == channel_id]
    
    if len(channel_rows) == 0:
        raise ValueError(f"Channel {channel_id} not found in metadata.")
    
    if len(channel_rows) > 1:
        raise ValueError(
            f"Channel {channel_id} has multiple metadata rows. "
            "Use cleaned metadata or handle duplicates first."
        )
    
    return channel_rows.iloc[0]


def load_channel_data(channel_id, train_dir, test_dir):
    train_path = os.path.join(train_dir, f"{channel_id}.npy")
    test_path = os.path.join(test_dir, f"{channel_id}.npy")
    
    if not os.path.exists(train_path):
        raise FileNotFoundError(f"Train file not found: {train_path}")
    
    if not os.path.exists(test_path):
        raise FileNotFoundError(f"Test file not found: {test_path}")
    
    train_data = np.load(train_path)
    test_data = np.load(test_path)
    
    return train_data, test_data


def create_binary_labels(test_length, anomaly_sequences):
    labels = np.zeros(test_length, dtype=int)
    
    for start, end in anomaly_sequences:
        labels[start:end + 1] = 1
    
    return labels


def validate_channel_alignment(test_data, test_labels, metadata_num_values):
    return {
        "test_length_matches_labels": test_data.shape[0] == len(test_labels),
        "test_length_matches_metadata": test_data.shape[0] == metadata_num_values
    }

## Step 5: Prepare One Channel

Before preparing multiple channels, I first prepare one known channel, `A-1`, to validate the preparation pipeline.

This step loads the raw train and test arrays, creates binary anomaly labels for the test data, and checks that the labels align with the test sequence length.

In [11]:
channel_id = "A-1"

metadata = get_channel_metadata(labels_clean_df, channel_id)
train_data, test_data = load_channel_data(channel_id, train_dir, test_dir)

anomaly_sequences = metadata["anomaly_sequences_parsed"]
test_labels = create_binary_labels(test_data.shape[0], anomaly_sequences)

alignment_checks = validate_channel_alignment(
    test_data=test_data,
    test_labels=test_labels,
    metadata_num_values=metadata["num_values"]
)

print("Channel:", channel_id)
print("Spacecraft:", metadata["spacecraft"])
print("Train shape:", train_data.shape)
print("Test shape:", test_data.shape)
print("Labels shape:", test_labels.shape)
print("Anomaly intervals:", anomaly_sequences)
print("Anomalous time steps:", test_labels.sum())
print("Anomaly percentage:", round(test_labels.mean() * 100, 2), "%")
print("Alignment checks:", alignment_checks)

Channel: A-1
Spacecraft: SMAP
Train shape: (2880, 25)
Test shape: (8640, 25)
Labels shape: (8640,)
Anomaly intervals: [[4690, 4774]]
Anomalous time steps: 85
Anomaly percentage: 0.98 %
Alignment checks: {'test_length_matches_labels': True, 'test_length_matches_metadata': np.True_}


## Step 6: Data Quality Checks

Before scaling or modeling, I check the train and test arrays for basic data quality issues.

This includes:

- Missing values
- Infinite values
- Constant features

Constant features are important because they do not provide useful variation for many models and can affect preprocessing decisions.

In [12]:
def check_data_quality(data, dataset_name):
    data_2d = data.reshape(-1, 1) if data.ndim == 1 else data
    
    quality = {
        "dataset": dataset_name,
        "shape": data.shape,
        "has_nan": np.isnan(data_2d).any(),
        "num_nan": int(np.isnan(data_2d).sum()),
        "has_inf": np.isinf(data_2d).any(),
        "num_inf": int(np.isinf(data_2d).sum()),
        "num_features": data_2d.shape[1],
        "num_constant_features": int((data_2d.std(axis=0) == 0).sum())
    }
    
    return quality

In [13]:
train_quality = check_data_quality(train_data, "train")
test_quality = check_data_quality(test_data, "test")

quality_df = pd.DataFrame([train_quality, test_quality])
quality_df

,dataset,shape,has_nan,num_nan,has_inf,num_inf,num_features,num_constant_features
0,train,"(2880, 25)",False,0,False,0,25,6
1,test,"(8640, 25)",False,0,False,0,25,4


The data quality check for channel `A-1` shows that both train and test data contain no missing values and no infinite values.

The train data has 6 constant features, while the test data has 4 constant features. Constant features have no variation within a dataset. However, in anomaly detection, a feature that is constant during training but changes during testing may still be useful because the change itself could indicate abnormal behavior.

Therefore, I will not remove constant features at this stage. I will keep them for the first preprocessing version and proceed with scaling.

In [14]:
train_2d = train_data.reshape(-1, 1) if train_data.ndim == 1 else train_data
test_2d = test_data.reshape(-1, 1) if test_data.ndim == 1 else test_data

train_constant_features = np.where(train_2d.std(axis=0) == 0)[0]
test_constant_features = np.where(test_2d.std(axis=0) == 0)[0]

print("Constant features in train:", train_constant_features)
print("Constant features in test:", test_constant_features)

print("Constant in both train and test:", sorted(set(train_constant_features).intersection(set(test_constant_features))))
print("Constant only in train:", sorted(set(train_constant_features) - set(test_constant_features)))
print("Constant only in test:", sorted(set(test_constant_features) - set(train_constant_features)))

Constant features in train: [10 12 15 16 23 24]
Constant features in test: [15 16 23 24]
Constant in both train and test: [np.int64(15), np.int64(16), np.int64(23), np.int64(24)]
Constant only in train: [np.int64(10), np.int64(12)]
Constant only in test: []


## Step 7: Scaling

Feature scaling is important for many anomaly detection models because feature magnitudes can affect distance-based methods, reconstruction error, and optimization.

To avoid data leakage, the scaler is fitted only on the training data. The same fitted scaler is then used to transform both training and test data.

The test data should not influence the fitted scaling parameters.

In [15]:
scaler = StandardScaler()

train_2d = train_data.reshape(-1, 1) if train_data.ndim == 1 else train_data
test_2d = test_data.reshape(-1, 1) if test_data.ndim == 1 else test_data

train_scaled = scaler.fit_transform(train_2d)
test_scaled = scaler.transform(test_2d)

print("Train scaled shape:", train_scaled.shape)
print("Test scaled shape:", test_scaled.shape)

print("\nTrain scaled mean, first 5 features:")
print(train_scaled.mean(axis=0)[:5])

print("\nTrain scaled std, first 5 features:")
print(train_scaled.std(axis=0)[:5])

Train scaled shape: (2880, 25)
Test scaled shape: (8640, 25)

Train scaled mean, first 5 features:
[ 4.42978987e-14  1.29757316e-16  1.93758975e-16 -3.60533362e-16
 -3.34228208e-16]

Train scaled std, first 5 features:
[0. 1. 1. 1. 1.]



After applying `StandardScaler`, the training data has approximately zero mean for the first features. Feature `0` has a scaled standard deviation of `0`, which means it is constant in the training data.

This is not an error. A constant feature has no variation in the training set, so after scaling it becomes all zeros in the scaled training data.

In anomaly detection, I will not remove constant features at this stage because a feature that is constant during training but changes during testing may still be useful for detecting abnormal behavior.

## Step 8: Create a Reusable Channel Preparation Function

After validating the preparation and scaling steps on channel `A-1`, I will wrap the logic into a reusable function.

This function will:

1. Retrieve the channel metadata.
2. Load the train and test `.npy` files.
3. Create binary anomaly labels for the test data.
4. Validate test-label alignment.
5. Check basic data quality.
6. Fit a scaler on the training data only.
7. Transform both train and test data using the fitted scaler.

This makes the preprocessing pipeline reusable for any selected channel.

In [16]:
def prepare_channel(channel_id, labels_df, train_dir, test_dir, scaler_class=StandardScaler):
    # 1. Get metadata
    metadata = get_channel_metadata(labels_df, channel_id)
    
    # 2. Load train and test data
    train_data, test_data = load_channel_data(channel_id, train_dir, test_dir)
    
    # 3. Convert to 2D if needed
    train_2d = train_data.reshape(-1, 1) if train_data.ndim == 1 else train_data
    test_2d = test_data.reshape(-1, 1) if test_data.ndim == 1 else test_data
    
    # 4. Create test labels
    anomaly_sequences = metadata["anomaly_sequences_parsed"]
    test_labels = create_binary_labels(test_2d.shape[0], anomaly_sequences)
    
    # 5. Validate alignment
    alignment_checks = validate_channel_alignment(
        test_data=test_2d,
        test_labels=test_labels,
        metadata_num_values=metadata["num_values"]
    )
    
    # 6. Data quality checks
    train_quality = check_data_quality(train_2d, "train")
    test_quality = check_data_quality(test_2d, "test")
    
    # 7. Scale data
    scaler = scaler_class()
    train_scaled = scaler.fit_transform(train_2d)
    test_scaled = scaler.transform(test_2d)
    
    # 8. Package results
    prepared = {
        "channel_id": channel_id,
        "spacecraft": metadata["spacecraft"],
        "metadata": metadata,
        "train_data": train_2d,
        "test_data": test_2d,
        "test_labels": test_labels,
        "train_scaled": train_scaled,
        "test_scaled": test_scaled,
        "scaler": scaler,
        "alignment_checks": alignment_checks,
        "quality_df": pd.DataFrame([train_quality, test_quality]),
        "anomaly_sequences": anomaly_sequences,
        "anomaly_classes": metadata["class_parsed"],
        "anomaly_percentage": round(test_labels.mean() * 100, 2)
    }
    
    return prepared

## Step 9: Test the Preparation Function on `A-1`

I will test the reusable preparation function on `A-1` to confirm that it produces the same outputs as the manual preparation steps.

In [17]:
prepared_a1 = prepare_channel(
    channel_id="A-1",
    labels_df=labels_clean_df,
    train_dir=train_dir,
    test_dir=test_dir
)

print("Channel:", prepared_a1["channel_id"])
print("Spacecraft:", prepared_a1["spacecraft"])
print("Train scaled shape:", prepared_a1["train_scaled"].shape)
print("Test scaled shape:", prepared_a1["test_scaled"].shape)
print("Test labels shape:", prepared_a1["test_labels"].shape)
print("Anomaly sequences:", prepared_a1["anomaly_sequences"])
print("Anomaly classes:", prepared_a1["anomaly_classes"])
print("Anomaly percentage:", prepared_a1["anomaly_percentage"], "%")
print("Alignment checks:", prepared_a1["alignment_checks"])

prepared_a1["quality_df"]

Channel: A-1
Spacecraft: SMAP
Train scaled shape: (2880, 25)
Test scaled shape: (8640, 25)
Test labels shape: (8640,)
Anomaly sequences: [[4690, 4774]]
Anomaly classes: ['point']
Anomaly percentage: 0.98 %
Alignment checks: {'test_length_matches_labels': True, 'test_length_matches_metadata': np.True_}


,dataset,shape,has_nan,num_nan,has_inf,num_inf,num_features,num_constant_features
0,train,"(2880, 25)",False,0,False,0,25,6
1,test,"(8640, 25)",False,0,False,0,25,4


## Step 10: Prepare Selected SMAP Channels

After validating the preparation function on `A-1`, I will apply it to a small set of selected SMAP channels.

For the first modeling version, I will focus on SMAP channels because they share the same feature dimension of 25 features per time step.

In [ ]:
# Choose preparation scope:
# "selected"  -> prepare only manually selected channels
# "smap"      -> prepare all clean SMAP channels, excluding duplicated channels like P-2
# "all_clean" -> prepare all clean channels, SMAP + MSL, excluding duplicated channels

preparation_scope = "selected"  # options: "selected", "smap", "all_clean"

if preparation_scope == "selected":
    selected_channels = ["A-1", "A-2", "P-1", "G-7"]

elif preparation_scope == "smap":
    selected_channels = labels_clean_df[
        labels_clean_df["spacecraft"] == "SMAP"
    ]["chan_id"].tolist()

elif preparation_scope == "all_clean":
    selected_channels = labels_clean_df["chan_id"].tolist()

else:
    raise ValueError("Invalid preparation_scope. Choose from: 'selected', 'smap', or 'all_clean'.")

prepared_channels = {}

for channel_id in selected_channels:
    prepared_channels[channel_id] = prepare_channel(
        channel_id=channel_id,
        labels_df=labels_clean_df,
        train_dir=train_dir,
        test_dir=test_dir
    )

print("Preparation scope:", preparation_scope)
print("Number of selected channels:", len(selected_channels))
print("Number of prepared channels:", len(prepared_channels))
print("First 10 prepared channels:", list(prepared_channels.keys())[:10])

ValueError: Invalid preparation_scope. Choose from: 'selected', 'smap', or 'all_clean'.

In [20]:
summary_rows = []

for channel_id, prepared in prepared_channels.items():
    summary_rows.append({
        "channel_id": channel_id,
        "spacecraft": prepared["spacecraft"],
        "train_shape": prepared["train_scaled"].shape,
        "test_shape": prepared["test_scaled"].shape,
        "labels_shape": prepared["test_labels"].shape,
        "anomaly_sequences": prepared["anomaly_sequences"],
        "anomaly_classes": prepared["anomaly_classes"],
        "anomaly_percentage": prepared["anomaly_percentage"],
        "alignment_ok": all(prepared["alignment_checks"].values()),
        "train_constant_features": prepared["quality_df"].loc[
            prepared["quality_df"]["dataset"] == "train", 
            "num_constant_features"
        ].iloc[0],
        "test_constant_features": prepared["quality_df"].loc[
            prepared["quality_df"]["dataset"] == "test", 
            "num_constant_features"
        ].iloc[0]
    })

prepared_summary_df = pd.DataFrame(summary_rows)

prepared_summary_df

,channel_id,spacecraft,train_shape,test_shape,labels_shape,anomaly_sequences,anomaly_classes,anomaly_percentage,alignment_ok,train_constant_features,test_constant_features
0,A-1,SMAP,"(2880, 25)","(8640, 25)","(8640,)","[[4690, 4774]]",[point],0.98,True,6,4
1,A-2,SMAP,"(2648, 25)","(7914, 25)","(7914,)","[[4450, 4560]]",[contextual],1.40,True,7,7
2,P-1,SMAP,"(2872, 25)","(8505, 25)","(8505,)","[[2149, 2349], [4536, 4844], [3539, 3779]]","[contextual, contextual, contextual]",8.83,True,9,7
3,G-7,SMAP,"(2446, 25)","(8029, 25)","(8029,)","[[3650, 3750], [5050, 5100], [7560, 7675]]","[contextual, point, contextual]",3.34,True,24,14


The reusable preparation function was successfully applied to selected SMAP channels.

All selected channels passed the alignment check, meaning the generated binary labels match the test sequence lengths and metadata `num_values`.

The selected SMAP channels all have 25 features, which makes them suitable for the first modeling version with a consistent input dimension.

The anomaly percentage varies across channels. `A-1` and `A-2` have very rare anomalies, while `P-1` has a larger anomalous portion because it contains three anomaly intervals. This confirms that the dataset is imbalanced, so accuracy should not be used as the main evaluation metric.

The data quality checks also show that constant features are common. For the first preprocessing version, constant features will be kept because a feature that is constant during training but changes during testing may still be useful for anomaly detection.

## Step 11: Save Prepared Data

After preparing and scaling the selected channels, I save the processed arrays to the `data/processed/` folder.

For each prepared channel, I save:

- Scaled training data
- Scaled test data
- Binary test labels

I also save a summary CSV file containing the preparation metadata for the selected channels.

The processed files are generated artifacts and should not be committed to Git if `data/processed/` is ignored in `.gitignore`.

In [22]:
import os
import json

PROCESSED_DIR = "../data/processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

# Save arrays for each prepared channel
for channel_id, prepared in prepared_channels.items():
    np.save(
        os.path.join(PROCESSED_DIR, f"{channel_id}_train_scaled.npy"),
        prepared["train_scaled"]
    )
    
    np.save(
        os.path.join(PROCESSED_DIR, f"{channel_id}_test_scaled.npy"),
        prepared["test_scaled"]
    )
    
    np.save(
        os.path.join(PROCESSED_DIR, f"{channel_id}_test_labels.npy"),
        prepared["test_labels"]
    )

# Save preparation summary table
summary_path = os.path.join(PROCESSED_DIR, "prepared_summary.csv")
prepared_summary_df.to_csv(summary_path, index=False)

print("Saved processed data to:", PROCESSED_DIR)
print("Number of saved channels:", len(prepared_channels))
print("Saved summary file:", summary_path)

Saved processed data to: ../data/processed
Number of saved channels: 53
Saved summary file: ../data/processed\prepared_summary.csv
